In [ ]:
import os
import numpy as np
import tensorflow as tf
import keras_cv

def calcular_fid_carpetas(carpeta_reales, carpeta_generadas, tamano_lote=32):
    """
    Calcula el Fréchet Inception Distance (FID) entre dos carpetas de imágenes.
    """
    print("Cargando datasets para evaluar FID...")
    
    # 1. Cargar las imágenes reales desde disco
    ds_reales = tf.keras.utils.image_dataset_from_directory(
        carpeta_reales,
        label_mode=None,
        image_size=(64, 64),
        batch_size=tamano_lote,
        shuffle=False
    ).map(lambda x: tf.cast(x, tf.float32)) # InceptionV3 espera floats (o se encarga de normalizar)

    # 2. Cargar las imágenes generadas desde disco
    ds_generadas = tf.keras.utils.image_dataset_from_directory(
        carpeta_generadas,
        label_mode=None,
        image_size=(64, 64),
        batch_size=tamano_lote,
        shuffle=False
    ).map(lambda x: tf.cast(x, tf.float32))

    # 3. Inicializar la métrica FID de KerasCV (utiliza InceptionV3 internamente)
    fid_metrica = keras_cv.metrics.FrechetInceptionDistance()

    print("Calculando activaciones de InceptionV3 para imágenes reales...")
    for lote in ds_reales:
        # KerasCV espera imágenes en rango [0, 255]
        fid_metrica.update_state(lote, real=True)

    print("Calculando activaciones de InceptionV3 para imágenes generadas...")
    for lote in ds_generadas:
        fid_metrica.update_state(lote, real=False)

    # 4. Obtener el resultado final
    score_fid = fid_metrica.result().numpy()
    print(f"\n✨ Fréchet Inception Distance (FID): {score_fid:.4f}")
    
    return score_fid

# =====================================================================
# EJEMPLO DE USO:
# =====================================================================
RUTA_REALES = "data/images" 
RUTA_GENERADAS = "Submission/imgs_ae" 
score = calcular_fid_carpetas(RUTA_REALES, RUTA_GENERADAS)